In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import re

university_systems = [
#    "University of Alabama System", "University of Alaska System", "University of Arizona System", "University of Arkansas System",
#    "University of California System", "University of Central Florida System", "University of Colorado System", "University of Connecticut System",
#    "University of Florida System", "University of Georgia System", "University of Hawaii System", "University of Washington System",
#    "University of Idaho System", "University of Illinois System", "University of Indiana System", "University of Iowa System",
#    "University of Kansas System", "University of Kentucky System", "University of Louisiana System", "University of Maine System",
#    "University of Maryland System", "University of Massachusetts System", "University of Michigan System", "University of Minnesota System",
#    "University of Mississippi System", "University of Missouri System", "University of Montana System", "University of Nebraska System",
#    "University of Nevada System", "University of New Hampshire System", "University of New Mexico System", "University of North Carolina System",
#    "University of North Dakota System", "University of North Florida System", "University of Oklahoma System", "University of Virginia System",
#    "University of Pennsylvania System", "University of Rhode Island System", "University of South Carolina System", "University of South Dakota System",
#    "University of South Florida System", "University of Southern Mississippi System", "University of Tennessee System", "University of Texas System",
#    "University of Utah System", "University of Vermont System", "University of West Florida System", "University of Wisconsin System",
#    "University of Wyoming System", "California State University System", "City University of New York System", "Indiana University System",
#    "Pennsylvania State System of Higher Education", "Purdue University System", "State University of New York System", "Texas A&M University System",
#    "West Virginia University System"
]

def clean_c3_field(c3_content):
    if pd.isna(c3_content):
        return ''

    items = [item.strip() for item in c3_content.split(';') if item.strip() and item.strip() not in university_systems]
    return '; '.join(items)

def extract_countries(addresses):

    if pd.isna(addresses):
        return ''

    country_list = []
    addresses_list = addresses.split('|')

    for address in addresses_list:
        address = address.strip()
        if address:
            parts = address.split(',')
            country = parts[-1].strip()[:-1]
            if "USA" in country:
                country = "USA"
            if "Turkey" in country:
                country = "Turkiye"
            country_list.append(country)

    return ' ; '.join(country_list)

def parse_wos_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as file:
        lines = file.readlines()

    data, record = [], {}
    current_field = None

    field_mapping = {
        "PT": "start_new_record",
        "AB": "Abstract",
        "AF": "Authors",
        "TI": "Title",
        "SO": "Journal",
        "PY": "Year",
        "C1": "Addresses",
        "DI": "DOI",
        "DE": "Keywords",
        "TC": "Cite",
        "C3": "C3",
        "FU": "Funding",
        "WC": "Subject_Category"
    }

    multi_line_fields = ['AF', 'C3', 'WC', "DE", "TI", "FU", "C1"]

    continuation_logic = {
        'Authors': lambda prev, line: prev + '|' + line.strip(),
        'C3': lambda prev, line: prev + ' ' + line.strip(),
        'Subject_Category': lambda prev, line: prev + ' ' + line.strip(),
        "Keywords": lambda prev, line: prev + ' ' + line.strip(),
        "Title": lambda prev, line: prev + ' ' + line.strip(),
        "Funding": lambda prev, line: prev + ' ' + line.strip(),
        "Addresses": lambda prev, line: prev + '|' + line.strip()
    }

    for line in lines:
        if len(line.strip()) == 0:
            continue

        if len(line) < 3:
            continue

        prefix, content = line[:2], line[3:].strip()

        if prefix in field_mapping:
            if prefix == "PT":
                if record:
                    data.append(record)
                record = {
                    'index': len(data) + 1,
                    'Abstract': '',
                    'Authors': '',
                    'C3': '',
                    'DOI': '',
                    'Funding': '',
                    'Journal': '',
                    'ISI_Keywords': '',
                    'Keywords': '',
                    'Year': '',
                    'Addresses': '',
                    'Title': '',
                    'Cite': '',
                    'Subject_Category': ''
                }
                current_field = None
            else:
                record[field_mapping[prefix]] = content
                current_field = field_mapping[prefix] if prefix in multi_line_fields else None
        else:
            if line.startswith(" "):
                if current_field in continuation_logic:
                    record[current_field] = continuation_logic[current_field](record[current_field], line)
            else:
                current_field = None

    if record:
        data.append(record)

    df = pd.DataFrame(data)

    df['Authors'] = df['Authors'].str.rstrip('|')
    df['Journal'] = df['Journal'].str.split(',').str[0]

    df['C3'] = df['C3'].apply(clean_c3_field)

    df['Countries'] = df['Addresses'].apply(extract_countries)

    return df

def save_to_excel(df, output_filepath):
    df.to_excel(output_filepath, index=False, engine='openpyxl')

def process_wos_to_excel(wos_filepath, output_filepath):
    df = parse_wos_file(wos_filepath)
    save_to_excel(df, output_filepath)

if __name__ == "__main__":
    wos_filepath = '/content/drive/MyDrive/HPC/merged.txt'
    output_filepath = 'output_wos_file.xlsx'
    process_wos_to_excel(wos_filepath, output_filepath)


In [ ]:
import pandas as pd

excel_file = './output_wos_file.xlsx'
df = pd.read_excel(excel_file)

In [ ]:
import pandas as pd
import networkx as nx

G = nx.Graph()

for index, row in df.iterrows():
    if isinstance(row['Countries'], str):
        Countries = [country.strip() for country in row['Countries'].split(';') if country.strip()]

        for i in range(len(Countries)):
            for j in range(i + 1, len(Countries)):
                if Countries[i] and Countries[j]:
                    if G.has_edge(Countries[i], Countries[j]):
                        G[Countries[i]][Countries[j]]['weight'] += 1
                    else:
                        G.add_edge(Countries[i], Countries[j], weight=1)

G = nx.relabel_nodes(G, lambda x: x.strip() if isinstance(x, str) else x)

nx.write_graphml(G, 'co_Countries_network.graphml')

print("Network saved as co_Countries_network.graphml")


In [ ]:
import networkx as nx
import community.community_louvain as community_louvain
from networkx.algorithms.community import greedy_modularity_communities
from networkx.algorithms import approximation as approx


def load_graph(graphml_file):
    G = nx.read_graphml(graphml_file)
    G.remove_edges_from(nx.selfloop_edges(G))  # Remove self-loops
    return G


def calculate_modularity_and_communities(G):
    partition = community_louvain.best_partition(G)
    modularity = community_louvain.modularity(partition, G)
    return partition, modularity


def label_communities(G, partition):
    for node, community in partition.items():
        G.nodes[node]['community'] = community

def find_cliques(G):
    cliques = list(nx.find_cliques(G))
    return cliques

def detect_core_periphery(G):
    core_nodes = nx.k_core(G).nodes()
    periphery_nodes = [node for node in G.nodes() if node not in core_nodes]
    return core_nodes, periphery_nodes

def calculate_betweenness_centrality(G):
    betweenness = nx.betweenness_centrality(G)
    return betweenness

def detect_communities(G):
    communities = greedy_modularity_communities(G)
    return communities

def save_graph_with_communities(G, output_file):
    nx.write_graphml(G, output_file)
    print(f"Graph saved with community labels in {output_file}")

def analyze_graph(graphml_file, output_file):
    G = load_graph(graphml_file)

    partition, modularity = calculate_modularity_and_communities(G)
    print(f"Modularity: {modularity}")
    print(f"Communities detected (using Louvain): {len(set(partition.values()))}")

    label_communities(G, partition)

    cliques = find_cliques(G)
    print(f"Number of cliques found: {len(cliques)}")

    core_nodes, periphery_nodes = detect_core_periphery(G)
    print(f"Core nodes: {len(core_nodes)}, Periphery nodes: {len(periphery_nodes)}")

    betweenness = calculate_betweenness_centrality(G)
    top_bridging_nodes = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:5]
    print("Top 5 bridging nodes based on betweenness centrality:")
    for node, value in top_bridging_nodes:
        print(f"Node: {node}, Betweenness Centrality: {value}")


    save_graph_with_communities(G, output_file)

graphml_file = "/content/co_Countries_network.graphml"
output_file = "Countries_network_with_communities.graphml"
analyze_graph(graphml_file, output_file)

## این کد فایل متنی که از سایت که از وب او ساینس گرفته شده را به فایل اکسل تبدیل می کند که ستون ها موارد زیر را شامل می شوند

#Co-Authorship Network Analysis

# این کد شبکه هم‌نویسندگی را تولید می‌کند. پس از اجرای آن، یک شبکه تولید خواهد شد که نیاز است آن را دانلود کنید. با استفاده از گفی می‌توانید آن را باز کنید. پس از باز کردن شبکه، پیشنهاد می‌شود الگوریتم‌های مختلف مرتب‌سازی گراف را امتحان کنید؛ احتمالاً الگوریتم فورس اطلس ۲ بهترین نتیجه را خواهد داشت. پس از اعمال الگوریتم، اندازه گره‌ها را بر اساس مرکزیت درجه تنظیم کنید و مقیاس‌دهی کنید تا گره‌ها معنای بیشتری پیدا کنند. همچنین می‌توانید از بخش پایینی صفحه در گفی برچسب گره‌ها را قابل نمایش کنید و تنظیمات مربوط به اندازه و نوع فونت برچسب‌ها را تغییر دهید. در ادامه کدی قرار داده شده است که محاسبات را برای شما انجام می‌دهد. کافی است فایل خروجی را در آدرس مورد نظر قرار دهید و محاسبات کاکر و میکرو را انجام دهید و در گزارشتان استفاده کنید.
# موفق باشید

In [ ]:
import pandas as pd
import networkx as nx

G = nx.Graph()

for index, row in df.iterrows():

    authors_data = row['Authors']

    if isinstance(authors_data, str):
        authors = authors_data.split('|')
        authors = [author.strip() for author in authors]

        for i in range(len(authors)):
            for j in range(i + 1, len(authors)):
                if G.has_edge(authors[i], authors[j]):
                    G[authors[i]][authors[j]]['weight'] += 1
                else:
                    G.add_edge(authors[i], authors[j], weight=1)

nx.write_graphml(G, 'Co-authorship_network.graphml')

#  ساخت جاینت کامپوننت هم‌نویسندگی




In [ ]:
import pandas as pd
import networkx as nx

G = nx.Graph()


for index, row in df.iterrows():
    authors_data = row['Authors']


    if isinstance(authors_data, str):
        authors = authors_data.split('|')
        authors = [author.strip() for author in authors]

        for i in range(len(authors)):
            for j in range(i + 1, len(authors)):
                if G.has_edge(authors[i], authors[j]):
                    G[authors[i]][authors[j]]['weight'] += 1
                else:
                    G.add_edge(authors[i], authors[j], weight=1)


giant_component_nodes = max(nx.connected_components(G), key=len)


giant_component_subgraph = G.subgraph(giant_component_nodes).copy()

nx.write_graphml(giant_component_subgraph, './giant_component_authorship_network.graphml')

print("Giant component extracted and saved successfully.")


#  ساخت کامیونیتی های بهینه در نتورک هم‌نویسندگی

In [ ]:
import networkx as nx
import community.community_louvain as community_louvain
from networkx.algorithms.community import greedy_modularity_communities
from networkx.algorithms import approximation as approx


def load_graph(graphml_file):
    G = nx.read_graphml(graphml_file)
    G.remove_edges_from(nx.selfloop_edges(G))  # Remove self-loops
    return G


def calculate_modularity_and_communities(G):
    partition = community_louvain.best_partition(G)
    modularity = community_louvain.modularity(partition, G)
    return partition, modularity


def label_communities(G, partition):
    for node, community in partition.items():
        G.nodes[node]['community'] = community

def find_cliques(G):
    cliques = list(nx.find_cliques(G))
    return cliques

def detect_core_periphery(G):
    core_nodes = nx.k_core(G).nodes()
    periphery_nodes = [node for node in G.nodes() if node not in core_nodes]
    return core_nodes, periphery_nodes

def calculate_betweenness_centrality(G):
    betweenness = nx.betweenness_centrality(G)
    return betweenness

def detect_communities(G):
    communities = greedy_modularity_communities(G)
    return communities

def save_graph_with_communities(G, output_file):
    nx.write_graphml(G, output_file)
    print(f"Graph saved with community labels in {output_file}")

def analyze_graph(graphml_file, output_file):
    G = load_graph(graphml_file)

    partition, modularity = calculate_modularity_and_communities(G)
    print(f"Modularity: {modularity}")
    print(f"Communities detected (using Louvain): {len(set(partition.values()))}")

    label_communities(G, partition)

    cliques = find_cliques(G)
    print(f"Number of cliques found: {len(cliques)}")

    core_nodes, periphery_nodes = detect_core_periphery(G)
    print(f"Core nodes: {len(core_nodes)}, Periphery nodes: {len(periphery_nodes)}")

    betweenness = calculate_betweenness_centrality(G)
    top_bridging_nodes = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:5]
    print("Top 5 bridging nodes based on betweenness centrality:")
    for node, value in top_bridging_nodes:
        print(f"Node: {node}, Betweenness Centrality: {value}")


    save_graph_with_communities(G, output_file)

graphml_file = "./giant_component_authorship_network.graphml"
output_file = "giant_component_authorship_network_with_communities.graphml"
analyze_graph(graphml_file, output_file)

# Co-Affiliation Network Analysis

# این کد شبکه موسسات را تولید می‌کند. پس از اجرای آن، یک شبکه تولید خواهد شد که نیاز است آن را دانلود کنید. با استفاده از گفی می‌توانید آن را باز کنید. پس از باز کردن شبکه، پیشنهاد می‌شود الگوریتم‌های مختلف مرتب‌سازی گراف را امتحان کنید؛ احتمالاً الگوریتم فورس اطلس ۲ بهترین نتیجه را خواهد داشت. پس از اعمال الگوریتم، اندازه گره‌ها را بر اساس مرکزیت درجه تنظیم کنید و مقیاس‌دهی کنید تا گره‌ها معنای بیشتری پیدا کنند. همچنین می‌توانید از بخش پایینی صفحه در گفی برچسب گره‌ها را قابل نمایش کنید و تنظیمات مربوط به اندازه و نوع فونت برچسب‌ها را تغییر دهید. در ادامه کدی قرار داده شده است که محاسبات را برای شما انجام می‌دهد. کافی است فایل خروجی را در آدرس مورد نظر قرار دهید و محاسبات کاکر و میکرو را انجام دهید و در گزارشتان استفاده کنید.
# موفق باشید

In [ ]:
import pandas as pd
import networkx as nx



G = nx.Graph()

for index, row in df.iterrows():
    if isinstance(row['C3'], str):
        affiliations = row['C3'].split(';')
        affiliations = [affiliation.strip() for affiliation in affiliations]

        for i in range(len(affiliations)):
            for j in range(i + 1, len(affiliations)):
                if G.has_edge(affiliations[i], affiliations[j]):
                    G[affiliations[i]][affiliations[j]]['weight'] += 1
                else:
                    G.add_edge(affiliations[i], affiliations[j], weight=1)

nx.write_graphml(G, 'co_affiliation_network.graphml')

print("Network saved as co_affiliation_network.net")


#  ساخت جاینت کامپوننت موسسات

In [ ]:
import networkx as nx

def load_graph(graphml_file):
    G = nx.read_graphml(graphml_file)
    return G

def get_giant_component(G):
    if nx.is_directed(G):
        largest_component = max(nx.weakly_connected_components(G), key=len)
    else:
        largest_component = max(nx.connected_components(G), key=len)

    G_giant = G.subgraph(largest_component).copy()
    return G_giant

def save_giant_component(G_giant, output_file):
    nx.write_graphml(G_giant, output_file)

# Main function
def main(graphml_file, output_file):
    G = load_graph(graphml_file)

    G_giant = get_giant_component(G)

    print(f"Original graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")
    print(f"Giant component has {G_giant.number_of_nodes()} nodes and {G_giant.number_of_edges()} edges.")

    save_giant_component(G_giant, output_file)
    print(f"Giant component saved to {output_file}")

graphml_file = './co_affiliation_network.graphml'  # Input GraphML file
output_file = 'giant_component_affiliation_network.graphml'             # Output GraphML file with giant component
main(graphml_file, output_file)
print("end")

Original graph has 6155 nodes and 119995 edges.
Giant component has 6050 nodes and 119910 edges.
Giant component saved to giant_component_affiliation_network.graphml
end


# ساخت کامیونیتی های بهینه در نتورک موسسات

In [ ]:
import networkx as nx
import community.community_louvain as community_louvain
from networkx.algorithms.community import greedy_modularity_communities
from networkx.algorithms import approximation as approx


def load_graph(graphml_file):
    G = nx.read_graphml(graphml_file)
    G.remove_edges_from(nx.selfloop_edges(G))  # Remove self-loops
    return G


def calculate_modularity_and_communities(G):
    partition = community_louvain.best_partition(G)
    modularity = community_louvain.modularity(partition, G)
    return partition, modularity


def label_communities(G, partition):
    for node, community in partition.items():
        G.nodes[node]['community'] = community

def find_cliques(G):
    cliques = list(nx.find_cliques(G))
    return cliques

def detect_core_periphery(G):
    core_nodes = nx.k_core(G).nodes()
    periphery_nodes = [node for node in G.nodes() if node not in core_nodes]
    return core_nodes, periphery_nodes

def calculate_betweenness_centrality(G):
    betweenness = nx.betweenness_centrality(G)
    return betweenness

def detect_communities(G):
    communities = greedy_modularity_communities(G)
    return communities

def save_graph_with_communities(G, output_file):
    nx.write_graphml(G, output_file)
    print(f"Graph saved with community labels in {output_file}")

def analyze_graph(graphml_file, output_file):
    G = load_graph(graphml_file)

    partition, modularity = calculate_modularity_and_communities(G)
    print(f"Modularity: {modularity}")
    print(f"Communities detected (using Louvain): {len(set(partition.values()))}")

    label_communities(G, partition)

    cliques = find_cliques(G)
    print(f"Number of cliques found: {len(cliques)}")

    core_nodes, periphery_nodes = detect_core_periphery(G)
    print(f"Core nodes: {len(core_nodes)}, Periphery nodes: {len(periphery_nodes)}")

    betweenness = calculate_betweenness_centrality(G)
    top_bridging_nodes = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:5]
    print("Top 5 bridging nodes based on betweenness centrality:")
    for node, value in top_bridging_nodes:
        print(f"Node: {node}, Betweenness Centrality: {value}")


    save_graph_with_communities(G, output_file)

graphml_file = "./giant_component_affiliation_network.graphml"
output_file = "giant_component_affiliation_network_with_communities.graphml"
analyze_graph(graphml_file, output_file)
print("end")

# Co-Subject Category Network Analysis

# این کد شبکه سابجکت کتگوری را تولید می‌کند. پس از اجرای آن، یک شبکه تولید خواهد شد که نیاز است آن را دانلود کنید. با استفاده از گفی می‌توانید آن را باز کنید. پس از باز کردن شبکه، پیشنهاد می‌شود الگوریتم‌های مختلف مرتب‌سازی گراف را امتحان کنید؛ احتمالاً الگوریتم فورس اطلس ۲ بهترین نتیجه را خواهد داشت. پس از اعمال الگوریتم، اندازه گره‌ها را بر اساس مرکزیت درجه تنظیم کنید و مقیاس‌دهی کنید تا گره‌ها معنای بیشتری پیدا کنند. همچنین می‌توانید از بخش پایینی صفحه در گفی برچسب گره‌ها را قابل نمایش کنید و تنظیمات مربوط به اندازه و نوع فونت برچسب‌ها را تغییر دهید. در ادامه کدی قرار داده شده است که محاسبات را برای شما انجام می‌دهد. کافی است فایل خروجی را در آدرس مورد نظر قرار دهید و محاسبات کاکر و میکرو را انجام دهید و در گزارشتان استفاده کنید.
# موفق باشید

In [ ]:
import pandas as pd
import networkx as nx

G = nx.Graph()

for index, row in df.iterrows():
    if isinstance(row['Subject_Category'], str):
        Subject_Category = row['Subject_Category'].split(';')
        Subject_Category = [Subject_Category.strip() for Subject_Category in Subject_Category]

        for i in range(len(Subject_Category)):
            for j in range(i + 1, len(Subject_Category)):
                if G.has_edge(Subject_Category[i], Subject_Category[j]):
                    G[Subject_Category[i]][Subject_Category[j]]['weight'] += 1
                else:
                    G.add_edge(Subject_Category[i], Subject_Category[j], weight=1)

nx.write_graphml(G, 'co_Subject_Category_network.graphml')

print("Network saved as co_Subject_Category_network.net")

#  ساخت جاینت کامپوننت نتورک Subject Category

In [ ]:
import networkx as nx

def load_graph(graphml_file):
    G = nx.read_graphml(graphml_file)
    return G

def get_giant_component(G):
    if nx.is_directed(G):
        largest_component = max(nx.weakly_connected_components(G), key=len)
    else:
        largest_component = max(nx.connected_components(G), key=len)

    G_giant = G.subgraph(largest_component).copy()
    return G_giant

def save_giant_component(G_giant, output_file):
    nx.write_graphml(G_giant, output_file)

# Main function
def main(graphml_file, output_file):
    G = load_graph(graphml_file)

    G_giant = get_giant_component(G)

    print(f"Original graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")
    print(f"Giant component has {G_giant.number_of_nodes()} nodes and {G_giant.number_of_edges()} edges.")

    save_giant_component(G_giant, output_file)
    print(f"Giant component saved to {output_file}")

graphml_file = './co_Funding_network.graphml'  # Input GraphML file
output_file = 'giant_component_Funding_network.graphml'             # Output GraphML file with giant component
main(graphml_file, output_file)
print("end")

# ساخت کامیونیتی های بهینه در نتورک Subject Category

In [ ]:
import networkx as nx
import community.community_louvain as community_louvain
from networkx.algorithms.community import greedy_modularity_communities
from networkx.algorithms import approximation as approx


def load_graph(graphml_file):
    G = nx.read_graphml(graphml_file)
    G.remove_edges_from(nx.selfloop_edges(G))  # Remove self-loops
    return G


def calculate_modularity_and_communities(G):
    partition = community_louvain.best_partition(G)
    modularity = community_louvain.modularity(partition, G)
    return partition, modularity


def label_communities(G, partition):
    for node, community in partition.items():
        G.nodes[node]['community'] = community

def find_cliques(G):
    cliques = list(nx.find_cliques(G))
    return cliques

def detect_core_periphery(G):
    core_nodes = nx.k_core(G).nodes()
    periphery_nodes = [node for node in G.nodes() if node not in core_nodes]
    return core_nodes, periphery_nodes

def calculate_betweenness_centrality(G):
    betweenness = nx.betweenness_centrality(G)
    return betweenness

def detect_communities(G):
    communities = greedy_modularity_communities(G)
    return communities

def save_graph_with_communities(G, output_file):
    nx.write_graphml(G, output_file)
    print(f"Graph saved with community labels in {output_file}")

def analyze_graph(graphml_file, output_file):
    G = load_graph(graphml_file)

    partition, modularity = calculate_modularity_and_communities(G)
    print(f"Modularity: {modularity}")
    print(f"Communities detected (using Louvain): {len(set(partition.values()))}")

    label_communities(G, partition)

    cliques = find_cliques(G)
    print(f"Number of cliques found: {len(cliques)}")

    core_nodes, periphery_nodes = detect_core_periphery(G)
    print(f"Core nodes: {len(core_nodes)}, Periphery nodes: {len(periphery_nodes)}")

    betweenness = calculate_betweenness_centrality(G)
    top_bridging_nodes = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:5]
    print("Top 5 bridging nodes based on betweenness centrality:")
    for node, value in top_bridging_nodes:
        print(f"Node: {node}, Betweenness Centrality: {value}")


    save_graph_with_communities(G, output_file)

graphml_file = "./giant_component_Funding_network.graphml"
output_file = "giant_component_Funding_network_with_communities.graphml"
analyze_graph(graphml_file, output_file)
print("end")

# Keywords Network With Technologies

# این کد شبکه کلید واژه های تکنولوژی را تولید می‌کند. پس از اجرای آن، یک شبکه تولید خواهد شد که نیاز است آن را دانلود کنید. با استفاده از گفی می‌توانید آن را باز کنید. پس از باز کردن شبکه، پیشنهاد می‌شود الگوریتم‌های مختلف مرتب‌سازی گراف را امتحان کنید؛ احتمالاً الگوریتم فورس اطلس ۲ بهترین نتیجه را خواهد داشت. پس از اعمال الگوریتم، اندازه گره‌ها را بر اساس مرکزیت درجه تنظیم کنید و مقیاس‌دهی کنید تا گره‌ها معنای بیشتری پیدا کنند. همچنین می‌توانید از بخش پایینی صفحه در گفی برچسب گره‌ها را قابل نمایش کنید و تنظیمات مربوط به اندازه و نوع فونت برچسب‌ها را تغییر دهید. در ادامه کدی قرار داده شده است که محاسبات را برای شما انجام می‌دهد. کافی است فایل خروجی را در آدرس مورد نظر قرار دهید و محاسبات کاکر و میکرو را انجام دهید و در گزارشتان استفاده کنید.
# موفق باشید

In [ ]:
import pandas as pd
import re
import spacy
import nltk
import networkx as nx
import matplotlib.pyplot as plt
from nltk.corpus import stopwords



nlp = spacy.load("en_core_web_sm")
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

with open('/content/drive/MyDrive/SNA/technologies.txt', 'r') as f:
    technology_list = [line.strip().lower() for line in f]


def preprocess_keywords(text):
    if not isinstance(text, str):
        return []
    # Step 1: Remove anything inside parentheses (e.g., "machine learning (ml)" -> "machine learning")
    text = re.sub(r'\([^)]*\)', '', text).strip()

    # Step 2: Split by | separator to get individual keywords
    keywords = text.split(';')

    cleaned_keywords = []

    for keyword in keywords:
        keyword = keyword.lower().strip()  # Convert to lowercase and remove extra spaces

        # Step 3: Remove stopwords
        keyword_tokens = [word for word in keyword.split() if word not in stop_words]

        # Step 4: Remove standalone abbreviations (e.g., "CNN")
        keyword_cleaned = re.sub(r'\b[A-Z]{2,}\b', '', ' '.join(keyword_tokens)).strip()

        # Step 5: Lemmatize the words to convert plural to singular
        doc = nlp(keyword_cleaned)
        keyword_lemmatized = ' '.join([token.lemma_ for token in doc])

        # Step 6: Keep non-empty keywords (numeric keywords will be removed after the network creation)
        if keyword_lemmatized and not re.search(r'\d', keyword_lemmatized):  # Ensure no numbers are present
            cleaned_keywords.append(keyword_lemmatized)

    return list(set(cleaned_keywords))  # Remove duplicate keywords


df['Processed Keywords'] = df['Keywords'].apply(preprocess_keywords)

G = nx.Graph()

for keywords in df['Processed Keywords']:
    keywords = [keyword for keyword in keywords if keyword and not re.search(r'\d', keyword)]

    for i in range(len(keywords)):
        for j in range(i + 1, len(keywords)):
            if G.has_edge(keywords[i], keywords[j]):
                G[keywords[i]][keywords[j]]['weight'] += 1
            else:
                G.add_edge(keywords[i], keywords[j], weight=1)

            if keywords[i] in technology_list:
                G.nodes[keywords[i]]['type'] = 'technology'
            else:
                G.nodes[keywords[i]]['type'] = 'general'

            if keywords[j] in technology_list:
                G.nodes[keywords[j]]['type'] = 'technology'
            else:
                G.nodes[keywords[j]]['type'] = 'general'

nodes_to_remove = [node for node in G.nodes() if node.isdigit() or node == '']
G.remove_nodes_from(nodes_to_remove)

plt.figure(figsize=(10, 8))

pos = nx.spring_layout(G, seed=42)

node_colors = ['lightblue' if G.nodes[node]['type'] == 'technology' else 'lightgreen' for node in G.nodes()]

nx.draw(G, pos, with_labels=True, node_size=500, node_color=node_colors, font_size=10, font_weight="bold", edge_color="gray")

plt.title("Keyword Network Graph with Technology Terms", fontsize=16)

plt.show()

nx.write_graphml(G, 'keywords_network_with_technologies.graphml')

# ساخت جاینت کامپوننت در نتورک کلید واژه های تکنولوژی

In [ ]:
import networkx as nx

def load_graph(graphml_file):
    G = nx.read_graphml(graphml_file)
    return G

def get_giant_component(G):
    if nx.is_directed(G):
        largest_component = max(nx.weakly_connected_components(G), key=len)
    else:
        largest_component = max(nx.connected_components(G), key=len)

    G_giant = G.subgraph(largest_component).copy()
    return G_giant

def save_giant_component(G_giant, output_file):
    nx.write_graphml(G_giant, output_file)

# Main function
def main(graphml_file, output_file):
    G = load_graph(graphml_file)

    G_giant = get_giant_component(G)

    print(f"Original graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")
    print(f"Giant component has {G_giant.number_of_nodes()} nodes and {G_giant.number_of_edges()} edges.")

    save_giant_component(G_giant, output_file)
    print(f"Giant component saved to {output_file}")


graphml_file = "./keywords_network_with_technologies.graphml"  # Input GraphML file
output_file = "giant_component_keywords_network_with_technologies.graphml"  # Output GraphML file with giant component
main(graphml_file, output_file)
print("end")

# ساخت کامیونیتی های بهینه در نتورک کلید واژه های تکنولوژی

In [ ]:
import networkx as nx
import community.community_louvain as community_louvain
from networkx.algorithms.community import greedy_modularity_communities
from networkx.algorithms import approximation as approx


def load_graph(graphml_file):
    G = nx.read_graphml(graphml_file)
    G.remove_edges_from(nx.selfloop_edges(G))  # Remove self-loops
    return G


def calculate_modularity_and_communities(G):
    partition = community_louvain.best_partition(G)
    modularity = community_louvain.modularity(partition, G)
    return partition, modularity


def label_communities(G, partition):
    for node, community in partition.items():
        G.nodes[node]['community'] = community

def find_cliques(G):
    cliques = list(nx.find_cliques(G))
    return cliques

def detect_core_periphery(G):
    core_nodes = nx.k_core(G).nodes()
    periphery_nodes = [node for node in G.nodes() if node not in core_nodes]
    return core_nodes, periphery_nodes

def calculate_betweenness_centrality(G):
    betweenness = nx.betweenness_centrality(G)
    return betweenness

def detect_communities(G):
    communities = greedy_modularity_communities(G)
    return communities

def save_graph_with_communities(G, output_file):
    nx.write_graphml(G, output_file)
    print(f"Graph saved with community labels in {output_file}")

def analyze_graph(graphml_file, output_file):
    G = load_graph(graphml_file)

    partition, modularity = calculate_modularity_and_communities(G)
    print(f"Modularity: {modularity}")
    print(f"Communities detected (using Louvain): {len(set(partition.values()))}")

    label_communities(G, partition)

    cliques = find_cliques(G)
    print(f"Number of cliques found: {len(cliques)}")

    core_nodes, periphery_nodes = detect_core_periphery(G)
    print(f"Core nodes: {len(core_nodes)}, Periphery nodes: {len(periphery_nodes)}")

    betweenness = calculate_betweenness_centrality(G)
    top_bridging_nodes = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:5]
    print("Top 5 bridging nodes based on betweenness centrality:")
    for node, value in top_bridging_nodes:
        print(f"Node: {node}, Betweenness Centrality: {value}")


    save_graph_with_communities(G, output_file)

graphml_file = "./keywords_network_with_technologies.graphml"
output_file = "keywords_network_with_technologies_with_communities.graphml"
analyze_graph(graphml_file, output_file)
print("end")
